# PQL.Assert – Multi-Workspace Test Runner

This notebook iterates through a configurable list of Microsoft Fabric workspace GUIDs, discovers all semantic models in each workspace using **semantic-link**, retrieves the PQL.Assert test functions defined in each model using `PQL.Assert.RetrieveTestsByEnvironmentV2`, and executes every test—applying user impersonation where specified by the `PQLAssert_ImpersonatedUserName` annotation (via Service XMLA `EffectiveUserName=`).

All test results are collected into a single output table that includes the workspace name, semantic model name, test function name, and the standard PQL.Assert columns (`TestName`, `Expected`, `Actual`, `Passed`).

## Requirements
- Run this notebook inside a **Microsoft Fabric** environment (Lakehouse or Warehouse attached, or standalone notebook)
- The `semantic-link-labs` package (installed in Step 2)
- The **PQL.Assert** library loaded into every target semantic model (`functions.tmdl` imported and model refreshed)
- The notebook identity (or the capacity admin token) must have **at least Build access** on each target workspace
- For impersonated tests the workspace must use **Row-Level Security (RLS)** and the target user must exist in the tenant


## Step 1 – Install dependencies

In [1]:
# Install semantic-link-labs (includes sempy.fabric)
%pip install semantic-link-labs

StatementMeta(, e14d21c5-a4c8-482e-9f3f-ce26246d8001, 8, Finished, Available, Finished, False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.1/875.1 kB 11.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 25.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 40.1 MB/s eta 0:00:00
  Attempting uninstall: azure-core
    Found existing installation: azure-core 2024.9.1
    Not uninstalling azure-core at /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages, outside environment /nfs4/pyenv-38cb6b48-9835-4d5d-98ce-fdd82838b1f8
    Can't uninstall 'azure-core'. No files were found to uninstall.
  At

## Step 2 – Imports

In [2]:
import sempy.fabric as fabric
import sempy_labs as labs
import pandas as pd

StatementMeta(, e14d21c5-a4c8-482e-9f3f-ce26246d8001, 10, Finished, Available, Finished, False)

## Step 3 – Configuration

Replace the placeholder GUIDs with the real workspace GUIDs you want to scan, and set `ENVIRONMENT` to the test environment you want to run (`DEV`, `TEST`, `PROD`, `ANY`, or `""` for all tests).

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# List every Fabric workspace GUID whose semantic models should be tested.
WORKSPACE_GUIDS: list[str] = [
    "00000000-0000-0000-0000-000000000002"  # Replace with your workspace GUID
]

# Environment filter passed to PQL.Assert.RetrieveTestsByEnvironmentV2.
# Options: "DEV" | "TEST" | "PROD" | "ANY" | "" (empty string = all tests)
ENVIRONMENT: str = "ANY"

StatementMeta(, e14d21c5-a4c8-482e-9f3f-ce26246d8001, 11, Finished, Available, Finished, False)

## Step 4 – Helper functions

In [4]:
def _retrieve_tests(workspace_id: str, dataset_id: str, environment: str) -> pd.DataFrame:
    """Call PQL.Assert.RetrieveTestsByEnvironmentV2 via XMLA (sempy.fabric.evaluate_dax).

    Returns a DataFrame with columns (with brackets):
        [Name]                           – function name (e.g. "DataQuality.ANY.Tests")
        [Description]                    – optional description annotation
        [PQLAssert_ImpersonatedUserName] – UPN to impersonate for RLS tests via Service XMLA (may be blank)
        [PQLAssert_RoleName]             – Role name for local RLS testing (not used in Fabric; local-only)

    Note: RetrieveTestsByEnvironmentV2 uses INFO.USERDEFINEDFUNCTIONS and
    INFO.ANNOTATIONS which require the XMLA endpoint. This is NOT compatible
    with the Power Automate "Execute Dataset Query" action; use
    PQL.Assert.RetrieveTestsByEnvironment (V1) for Power Automate flows.
    
    Note: [PQLAssert_RoleName] is for local Power BI Desktop testing only (Roles= connection string).
    Fabric notebooks always use EffectiveUserName= via the Service XMLA endpoint.
    """
    env_escaped = environment.replace('"', '""')  # escape any embedded quotes
    dax = f'EVALUATE PQL.Assert.RetrieveTestsByEnvironmentV2("{env_escaped}")'
    return fabric.evaluate_dax(workspace=workspace_id, dataset=dataset_id, dax_string=dax)


def _execute_test(workspace_id: str, dataset_id: str, test_name: str) -> pd.DataFrame:
    """Execute a PQL.Assert test function via sempy.fabric.evaluate_dax (XMLA).

    Returns a DataFrame with the standard PQL.Assert result columns:
        TestName, Expected, Actual, Passed
    """
    dax = f"EVALUATE {test_name}()"
    return fabric.evaluate_dax(workspace=workspace_id, dataset=dataset_id, dax_string=dax)


def _execute_test_as_user(
    workspace_id: str,
    dataset_id: str,
    test_name: str,
    username: str,
) -> pd.DataFrame:
    """Execute a PQL.Assert test function via sempy_labs.evaluate_dax_impersonation.

    Uses the semantic-link-labs REST API wrapper which handles the Power BI
    executeQueries endpoint with impersonation internally.

    Args:
        workspace_id: GUID of the Fabric workspace.
        dataset_id:   GUID of the semantic model (dataset).
        test_name:    Fully-qualified test function name (e.g. "RLS.ANY.Tests").
        username:     UPN of the user to impersonate (e.g. "user@contoso.com").

    Returns:
        DataFrame with TestName, Expected, Actual, Passed columns.
    """
    dax = f"EVALUATE {test_name}()"
    return labs.evaluate_dax_impersonation(
        dataset=dataset_id,
        dax_query=dax,
        user_name=username,
        workspace=workspace_id,
    )

StatementMeta(, e14d21c5-a4c8-482e-9f3f-ce26246d8001, 12, Finished, Available, Finished, False)

## Step 5 – Iterate workspaces, discover and execute tests

In [5]:
all_results: list[pd.DataFrame] = []

for workspace_id in WORKSPACE_GUIDS:

    # ── Resolve workspace display name ────────────────────────────────────────
    try:
        workspace_name = fabric.resolve_workspace_name(workspace_id)
    except Exception:
        workspace_name = workspace_id

    print(f"\n🗂  Workspace: {workspace_name} ({workspace_id})")

    # ── List semantic models using semantic-link (REST API) ───────────────────
    try:
        semantic_models: pd.DataFrame = fabric.list_datasets(
            workspace=workspace_id, 
            mode='rest', 
            endpoint='fabric'
        )
    except Exception as exc:
        print(f"  ⚠  Could not list datasets: {exc}")
        continue

    if semantic_models is None or semantic_models.empty:
        print("  ℹ  No semantic models found in this workspace.")
        continue

    for _, model_row in semantic_models.iterrows():
        dataset_id: str = str(model_row["Dataset Id"])
        dataset_name: str = str(model_row["Dataset Name"])

        print(f"\n  📊 Semantic Model: {dataset_name} ({dataset_id})")

        # ── Retrieve tests via PQL.Assert.RetrieveTestsByEnvironmentV2 ────────
        try:
            tests_df = _retrieve_tests(workspace_id, dataset_id, ENVIRONMENT)
        except Exception as exc:
            print(f"    ⚠  Could not retrieve tests (is PQL.Assert installed?): {exc}")
            continue

        if tests_df is None or tests_df.empty:
            print(f"    ℹ  No tests found for environment '{ENVIRONMENT}'.")
            continue

        print(f"    ✅ {len(tests_df)} test function(s) found for environment '{ENVIRONMENT}'.")

        # ── Execute each test ─────────────────────────────────────────────────
        for _, test_row in tests_df.iterrows():
            # Column names returned by evaluate_dax include brackets around them
            test_name: str = str(test_row.get("[Name]", test_row.iloc[0]))

            # Get impersonated user, handling pandas NA values properly
            raw_user = test_row.get("[PQLAssert_ImpersonatedUserName]", "")
            
            # Check if it's actually a valid username (not None, not NA, not empty)
            if pd.isna(raw_user) or raw_user is None or str(raw_user).strip() in ("", "<NA>"):
                impersonated_user = ""
            else:
                impersonated_user = str(raw_user).strip()

            try:
                if impersonated_user:
                    print(
                        f"    ▶ Executing '{test_name}' "
                        f"as '{impersonated_user}' (impersonated)…"
                    )
                    result_df = _execute_test_as_user(
                        workspace_id, dataset_id, test_name, impersonated_user
                    )
                else:
                    print(f"    ▶ Executing '{test_name}'…")
                    result_df = _execute_test(workspace_id, dataset_id, test_name)
            except Exception as exc:
                print(f"    ✗ Error executing '{test_name}': {exc}")
                # Create a failed test result for the error
                result_df = pd.DataFrame({
                    "[TestName]": [test_name],
                    "[Expected]": ["Success"],
                    "[Actual]": [f"Error: {str(exc)}"],
                    "[Passed]": [False]
                })

            # Annotate results with workspace / model context
            result_df.insert(0, "ImpersonatedUser", impersonated_user if impersonated_user else "")
            result_df.insert(0, "TestFunctionName", test_name)
            result_df.insert(0, "SemanticModelName", dataset_name)
            result_df.insert(0, "SemanticModelId", dataset_id)
            result_df.insert(0, "WorkspaceName", workspace_name)
            result_df.insert(0, "WorkspaceId", workspace_id)
            
            # Ensure ImpersonatedUser column stays as string type (not NA)
            result_df["ImpersonatedUser"] = result_df["ImpersonatedUser"].fillna("")

            all_results.append(result_df)

print("\n✅ Test execution complete.")

StatementMeta(, e14d21c5-a4c8-482e-9f3f-ce26246d8001, 13, Submitted, Running, Running, True)


🗂  Workspace: pqlint-demos [Test] (daeadd4c-c82a-493e-b610-a21a46a3e914)

  📊 Semantic Model: SampleModel (b7673338-4bc6-43e3-902f-328cd80aed0b)
    ✅ 1 test function(s) found for environment 'ANY'.
    ▶ Executing 'DataQuality.ANY.Tests'…

  📊 Semantic Model: TestingModel (944d9be9-6dbd-427d-9126-95f7701b342e)


## Step 6 – Display results

All results are combined into a single DataFrame. The summary line shows totals for passed and failed tests across all workspaces and models.

In [ ]:
if not all_results:
    print(
        "No test results were collected.\n"
        "Verify that:\n"
        "  1. WORKSPACE_GUIDS contains valid workspace GUIDs.\n"
        "  2. PQL.Assert is installed in the target semantic models.\n"
        "  3. The notebook has Build (or higher) access to each workspace."
    )
else:
    results_df = pd.concat(all_results, ignore_index=True)


    # Summary - check for column name with or without brackets
    total = len(results_df)
    passed_col = "[Passed]" if "[Passed]" in results_df.columns else "Passed"
    passed = int(results_df[passed_col].sum()) if passed_col in results_df.columns else 0
    failed = total - passed
    print(f"Results  |  Total: {total}  |  ✅ Passed: {passed}  |  ❌ Failed: {failed}")
    print()

    # Display the full results table
    display(results_df)

StatementMeta(, , -1, Waiting, , Waiting, True)